In [4]:
import numpy as np


class SVM:
    def __init__(self, kernel=lambda x, y: np.dot(x, y), eta=0.5, epochs=100):
        self.Alpha = None
        self.X = None
        self.y = None
        self.b = None
        self.eta = eta
        self.epochs = epochs
        self.kernel = kernel

    def dot_w(self, X):
        res = 0
        for alpha, y, X_vec in zip(self.Alpha, self.y, self.X):
            res += alpha * y * self.kernel(X_vec, X)
        return res

    def loss_prime(self, Alpha, X, y):
        del_l_alpha = []
        for k in range(len(Alpha)):
            val = 1
            for i in range(len(X)):
                val -= Alpha[i] * y[i] * y[k] * (self.kernel(X[i], X[k]))
            del_l_alpha.append(val)
        return np.array(del_l_alpha)

    def train(self, X, y):
        train_size, _ = X.shape
        self.b = 0
        Alpha = np.zeros(train_size)

        for _ in range(self.epochs):
            del_l_alpha = self.loss_prime(Alpha, X, y)
            Alpha = np.maximum(0, Alpha + self.eta * del_l_alpha)  #

        self.Alpha = Alpha
        self.X = X
        self.y = y
        interim = []
        for X_vec in self.X:
            interim.append(self.dot_w(X_vec))
        self.b = np.mean(y - interim)

    def classify(self, X):
        y = []
        for X_vec in X:
            y.append(self.dot_w(X_vec) + self.b)
        return np.sign(y)

In [5]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

df = pd.read_csv("heart.csv")
df = df.replace("?", np.nan).dropna()

Y = (df["condition"] > 0).astype(int).to_numpy()
Y = np.where(Y == 1, 1, -1)

X_df = df.drop(columns=["condition"])
X_df = pd.get_dummies(X_df, dtype=int)
X = X_df.to_numpy(dtype=float)

np.random.seed(42)
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.20, random_state=42, stratify=Y
)

# SVMs are sensitive to feature scales
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

svm = SVM(kernel=lambda x, y: np.dot(x, y), eta=0.001, epochs=1000)

svm.train(X_train, Y_train)

# Predict
svm_predictions = svm.classify(X_test)

svm_accuracy = accuracy_score(Y_test, svm_predictions)
svm_error = 1 - svm_accuracy

print(f"Accuracy : {svm_accuracy * 100:.2f}%")
print(f"Error    : {svm_error * 100:.2f}%")

Accuracy : 90.00%
Error    : 10.00%
